# A3.3 — Router Şablonlarını Başkalaştırma (Paraphrase) — Colab

Bu notebook **Google Colab'da**, ücretsiz T4 GPU ile çalışacak şekilde yazıldı.

**Ne yapıyor:** Router için üretilen ~24.999 sorunun **hepsini değil**,
yalnızca arkasındaki **~84 benzersiz şablonu** (yer tutucu içeren hâliyle)
büyük bir LLM'e (Qwen2.5-7B-Instruct) verip başkalaştırıyor. Başkalaştırılmış
şablonlar sonra yerelde (GPU'suz, saniyeler içinde) gerçek ürün/kategori/
tedarikçi adlarıyla yeniden çoğaltılıp tam veri setine dönüştürülüyor.

> ⚠️ **İlk tasarım hatası ve düzeltmesi:** İlk denemede 24.999 satırın HER
> BİRİNİ ayrı ayrı modele gönderiyorduk — 100 satır 357 saniye sürmüştü,
> tüm veri seti için ~25 saate karşılık geliyordu (Colab'ın ücretsiz oturum
> sınırı ~12 saat, bu yüzden imkânsızdı). Oysa 24.999 satırın tamamı yalnızca
> ~84 şablonun ürün/kategori/tedarikçi ile doldurulmuş hâli. Şimdi yalnızca
> o 84 şablonu başkalaştırıp, aynı çoğaltmayı yerelde tekrar uyguluyoruz —
> ~250x daha az model çağrısı, ~5 dakika.

**Neden ayrı bir model?** Kişi B'nin kuracağı Qwen2.5-**1.5B** üretimde
(routing + gerekçe yazımı) çalışacak küçük model. Bu notebook'taki
Qwen2.5-**7B** yalnızca **eğitim verisi hazırlarken bir kerelik** kullanılan,
çok daha güçlü bir model.

**Sıra:**
1. Runtime → Değiştir → Donanım hızlandırıcı → **T4 GPU** seç
2. `sablon_listesi.jsonl` dosyasını yükle (yerelde `uv run python -m
   training.build_dataset --sablon-ihrac-et data/egitim/sablon_listesi.jsonl`
   ile üretilir — yalnızca ~84 satır, çok küçük bir dosya)
3. Tüm şablonları tek seferde işle (küçük olduğu için örnekleme adımına
   gerek yok), sonucu indir


## 1. Ortamı hazırla

In [ ]:
!nvidia-smi


In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece


## 2. Şablon dosyasını yükle

Aşağıdaki hücre bir dosya seçme penceresi açacak — bilgisayarındaki `sablon_listesi.jsonl` dosyasını seç (24.999 satırlık `router_sorulari.jsonl` **DEĞİL** — bu çok daha küçük, ~84 satırlık şablon dosyası).

In [ ]:
from google.colab import files

yuklenen = files.upload()
sablon_dosyasi = next(iter(yuklenen.keys()))
print("Yüklenen dosya:", sablon_dosyasi)


## 3. Modeli yükle (4-bit, T4'e sığması için)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ADI = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ADI)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ADI,
    quantization_config=bnb_config,
    device_map="auto",
)
print("Model yüklendi.")


## 4. Başkalaştırma fonksiyonu

Üç güvenlik kontrolü var: **(1)** yer tutucu token (ör. `KATEGORI_ADI`) korunmamışsa o varyant atılır, **(2)** Çince/Korece/Japonca karaktere kayarsa atılır, **(3)** numaralandırma/madde işareti kalıntıları temizlenir. İkisi de başarısız olursa 2 kez tekrar dener, yine olmazsa o şablon için boş liste döner (bir sonraki hücrede orijinal şablon korunur, veri kaybı olmaz).

In [ ]:
import re

CJK_DESENI = re.compile(r"[\u4e00-\u9fff\u3040-\u30ff\uac00-\ud7af]")
NUMARALANDIRMA_DESENI = re.compile(r"^\s*\d+[\.\)\-]\s*")

PROMPT_SABLONU = '''Aşağıdaki Türkçe cümleyi, AYNI ANLAMI koruyarak {n} farklı şekilde yeniden yaz.

KURALLAR:
- Yalnızca TÜRKÇE yaz. Başka hiçbir dil kullanma (Çince, İngilizce vb. KESİNLİKLE YASAK).
- Cümledeki "{token}" kelimesini AYNEN, hiç değiştirmeden koru. O bir yer tutucudur —
  silme, çevirme veya başka bir kelimeyle değiştirme.
- Her varyantı ayrı satırda yaz. Satır başına numara, tire, madde işareti KOYMA.
- Başka hiçbir açıklama, giriş cümlesi, yorum yazma — yalnızca varyantları yaz.

Cümle: {sablon}'''


def parafraz_uret(
    sablon_metni: str, token: str | None, n: int = 2, max_new_tokens: int = 200, deneme: int = 2
) -> list[str]:
    token_ifadesi = token if token else "(yok — bu cümlede yer tutucu yok)"
    mesajlar = [
        {
            "role": "user",
            "content": PROMPT_SABLONU.format(sablon=sablon_metni, token=token_ifadesi, n=n),
        }
    ]
    girdi = tokenizer.apply_chat_template(mesajlar, tokenize=False, add_generation_prompt=True)
    girdi_id = tokenizer(girdi, return_tensors="pt").to(model.device)

    for _ in range(deneme):
        with torch.no_grad():
            cikti = model.generate(
                **girdi_id,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
            )
        metin = tokenizer.decode(
            cikti[0][girdi_id["input_ids"].shape[1] :], skip_special_tokens=True
        )
        satirlar = [
            NUMARALANDIRMA_DESENI.sub("", s).strip(" -•\t")
            for s in metin.split("\n")
            if s.strip()
        ]

        if token:
            satirlar = [s for s in satirlar if token in s]
        satirlar = [s for s in satirlar if not CJK_DESENI.search(s)]

        if satirlar:
            return satirlar[:n]

    return []  # guard: gecerli sonuc uretilemedi, bu sablon icin bos donuyor


## 5. Tüm şablonları işle

Yalnızca ~84 satır olduğu için örnekleme adımına gerek yok, hepsini tek seferde işleyip sonucu gözden geçiriyoruz.

In [ ]:
import json
import time

with open(sablon_dosyasi, encoding="utf-8") as f:
    sablonlar = [json.loads(satir) for satir in f]

print("Toplam şablon sayısı:", len(sablonlar))

t0 = time.time()
sonuclar = []
for kayit in sablonlar:
    varyantlar = parafraz_uret(kayit["sablon_metni"], kayit["placeholder_token"], n=2)
    sonuclar.append({**kayit, "parafraz_varyantlari": varyantlar})
t1 = time.time()

bos_sayisi = sum(1 for s in sonuclar if not s["parafraz_varyantlari"])
print(f"Süre: {t1 - t0:.1f} sn")
print(f"Guard tarafından reddedilen (boş kalan) şablon: {bos_sayisi}/{len(sonuclar)}")

print()
print("--- İlk 5 örnek ---")
for s in sonuclar[:5]:
    print("Orijinal:", s["sablon_metni"])
    print("Varyantlar:", s["parafraz_varyantlari"])
    print()


## 6. Kalite kontrolü (elle)

**Durup yukarıdaki örnekleri oku.** Varyantlar doğal mı, anlam korunuyor mu, yer tutucu (`KATEGORI_ADI` vb.) hâlâ metinde duruyor mu? Sorunluysa 4. hücredeki `PROMPT_SABLONU`'nu güncelleyip yeniden dene.

## 7. Sonucu kaydet ve indir

In [ ]:
from google.colab import files

CIKTI_DOSYASI = "sablon_parafraz.jsonl"

with open(CIKTI_DOSYASI, "w", encoding="utf-8") as f:
    for s in sonuclar:
        varyantlar = s["parafraz_varyantlari"] or [s["sablon_metni"]]
        for varyant in varyantlar:
            f.write(
                json.dumps(
                    {
                        "arac": s["arac"],
                        "varlik_turu": s["varlik_turu"],
                        "placeholder_token": s["placeholder_token"],
                        "sablon_metni": varyant,
                    },
                    ensure_ascii=False,
                )
                + "\n"
            )

print("Yazıldı:", CIKTI_DOSYASI)
files.download(CIKTI_DOSYASI)


## Sonrası

İndirdiğin `sablon_parafraz.jsonl` dosyasını projede
`data/egitim/sablon_parafraz.jsonl` olarak kaydet. Sonra yerelde şu
fonksiyonla tam veri setine (gerçek ürün/kategori/tedarikçi adlarıyla)
yeniden çoğaltılacak:

```python
import pandas as pd
from training.build_dataset import parafraz_sablonlarindan_veri_uret
from simulator.company import yapi_malzemesi_toptancisi
from simulator.catalog import katalog_uret

parafraz_df = pd.read_json("data/egitim/sablon_parafraz.jsonl", lines=True)
kataloglar = katalog_uret(yapi_malzemesi_toptancisi(), seed=42)
tam_veri = parafraz_sablonlarindan_veri_uret(
    parafraz_df, kataloglar["sku"], kataloglar["tedarikci"]
)
```
